# Phase 3: Feature Engineering

## Forecasting Objective
Predict the next 24 consecutive hours of PJM RTO electricity demand using historical load and calendar features.

## Forecasting Setup
- Forecast origin: Midnight Eastern Time
- Forecast horizon: 24 consecutive hours
- Target: Hourly electricity load (MW)
- Prediction method: Pooled direct multi-horizon regression
- Data leakage rule: Use only information available at the forecast origin

In [1]:
# Step 3.1: Load and prepare time-series data

from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd()
if not (project / "data_processed").exists():
    project = project.parent

file_path = project / "data_processed" / "pjm_rto_hourly_2021_2025.csv"

df = pd.read_csv(file_path)

# Parse UTC timestamps
df["datetime_utc"] = pd.to_datetime(
    df["datetime_beginning_utc"],
    utc=True
)

# Convert to Eastern Time
df["datetime_ept"] = df["datetime_utc"].dt.tz_convert(
    "America/New_York"
)

# Sort chronologically
df = df.sort_values("datetime_utc").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Start:", df["datetime_ept"].min())
print("End:", df["datetime_ept"].max())

display(df.head())

Dataset shape: (43824, 6)
Start: 2021-01-01 00:00:00-05:00
End: 2025-12-31 23:00:00-05:00


,datetime_beginning_utc,datetime_ept,mw,source_year,is_verified,datetime_utc
0,2021-01-01 05:00:00+00:00,2021-01-01 00:00:00-05:00,81842.644,2021,False,2021-01-01 05:00:00+00:00
1,2021-01-01 06:00:00+00:00,2021-01-01 01:00:00-05:00,80005.690,2021,False,2021-01-01 06:00:00+00:00
2,2021-01-01 07:00:00+00:00,2021-01-01 02:00:00-05:00,78940.704,2021,False,2021-01-01 07:00:00+00:00
3,2021-01-01 08:00:00+00:00,2021-01-01 03:00:00-05:00,78563.968,2021,False,2021-01-01 08:00:00+00:00
4,2021-01-01 09:00:00+00:00,2021-01-01 04:00:00-05:00,79136.453,2021,False,2021-01-01 09:00:00+00:00


In [2]:
# Step 3.2: Calendar Feature Engineering

# Create calendar features using Eastern Time
df["hour"] = df["datetime_ept"].dt.hour
df["day_of_week"] = df["datetime_ept"].dt.dayofweek
df["month"] = df["datetime_ept"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Extract Eastern calendar date
df["date_ept"] = df["datetime_ept"].dt.date

# Initial position within each local date
df["hour_position"] = df.groupby("date_ept").cumcount() + 1

print("Calendar features created!")

display(
    df[
        [
            "datetime_ept",
            "hour",
            "day_of_week",
            "month",
            "is_weekend",
            "hour_position"
        ]
    ].head(10)
)

print("\nHours per Eastern calendar day:")
print(df.groupby("date_ept").size().value_counts().sort_index())

Calendar features created!


,datetime_ept,hour,day_of_week,month,is_weekend,hour_position
0,2021-01-01 00:00:00-05:00,0,4,1,0,1
1,2021-01-01 01:00:00-05:00,1,4,1,0,2
2,2021-01-01 02:00:00-05:00,2,4,1,0,3
3,2021-01-01 03:00:00-05:00,3,4,1,0,4
4,2021-01-01 04:00:00-05:00,4,4,1,0,5
5,2021-01-01 05:00:00-05:00,5,4,1,0,6
6,2021-01-01 06:00:00-05:00,6,4,1,0,7
7,2021-01-01 07:00:00-05:00,7,4,1,0,8
8,2021-01-01 08:00:00-05:00,8,4,1,0,9
9,2021-01-01 09:00:00-05:00,9,4,1,0,10



Hours per Eastern calendar day:
23       5
24    1816
25       5
Name: count, dtype: int64


In [3]:
# Step 3.3: Create day-ahead forecasting samples

# Sort observations by UTC
df = df.sort_values("datetime_utc").reset_index(drop=True)

# Create lag features from historical load
df["lag_24"] = df["mw"].shift(24)
df["lag_48"] = df["mw"].shift(48)
df["lag_168"] = df["mw"].shift(168)

# Identify forecast origins at midnight Eastern Time
origin_indices = df.index[df["hour"] == 0].tolist()

# Create 24-hour forecasting windows
windows = []

for origin in origin_indices:

    # Require historical observations and a complete future window
    if origin < 168 or origin + 24 > len(df):
        continue

    window = df.iloc[origin:origin + 24].copy()

    # Forecast origin: the same for all 24 target hours
    window["forecast_origin"] = df.loc[origin, "datetime_utc"]

    # Forecast horizon: 1 through 24
    window["horizon"] = np.arange(1, 25)

    windows.append(window)

ml_df = pd.concat(windows, ignore_index=True)

print("ML dataset shape:", ml_df.shape)
print("Number of forecast origins:",
      ml_df["forecast_origin"].nunique())

display(
    ml_df[
        [
            "forecast_origin",
            "datetime_ept",
            "horizon",
            "hour",
            "lag_24",
            "lag_48",
            "lag_168",
            "mw"
        ]
    ].head(10)
)

ML dataset shape: (43656, 17)
Number of forecast origins: 1819


,forecast_origin,datetime_ept,horizon,hour,lag_24,lag_48,lag_168,mw
0,2021-01-08 05:00:00+00:00,2021-01-08 00:00:00-05:00,1,0,87118.924,85322.338,81842.644,88192.784
1,2021-01-08 05:00:00+00:00,2021-01-08 01:00:00-05:00,2,1,85351.434,83392.839,80005.690,86488.512
2,2021-01-08 05:00:00+00:00,2021-01-08 02:00:00-05:00,3,2,84762.492,82525.861,78940.704,85853.980
3,2021-01-08 05:00:00+00:00,2021-01-08 03:00:00-05:00,4,3,84939.037,82539.595,78563.968,85901.332
4,2021-01-08 05:00:00+00:00,2021-01-08 04:00:00-05:00,5,4,87058.963,84236.707,79136.453,87372.680
5,2021-01-08 05:00:00+00:00,2021-01-08 05:00:00-05:00,6,5,91165.498,88018.027,80865.889,91299.370
6,2021-01-08 05:00:00+00:00,2021-01-08 06:00:00-05:00,7,6,97929.816,94180.247,83088.263,97508.661
7,2021-01-08 05:00:00+00:00,2021-01-08 07:00:00-05:00,8,7,103430.585,99427.635,84833.878,103064.318
8,2021-01-08 05:00:00+00:00,2021-01-08 08:00:00-05:00,9,8,104186.336,101154.729,86033.226,105097.864
9,2021-01-08 05:00:00+00:00,2021-01-08 09:00:00-05:00,10,9,101763.767,101129.000,87796.165,105382.460


In [4]:
# Step 3.4: Validate forecasting windows and lag features

# 1. Every forecast origin must contain 24 targets
window_sizes = ml_df.groupby("forecast_origin").size()

print("All windows have 24 hours:",
      window_sizes.eq(24).all())

# 2. Horizons must be 1–24
valid_horizons = ml_df.groupby("forecast_origin")["horizon"].apply(
    lambda x: x.tolist() == list(range(1, 25))
)

print("All horizons valid:", valid_horizons.all())

# 3. Lag values must be available before forecast origin
for lag in [24, 48, 168]:
    lag_time = (
        ml_df["datetime_utc"]
        - pd.Timedelta(hours=lag)
    )

    valid = (lag_time < ml_df["forecast_origin"]).all()

    print(f"Lag {lag} leakage-free:", valid)

# 4. Check missing values
print("\nMissing lag values:")
print(ml_df[["lag_24", "lag_48", "lag_168"]].isna().sum())

# 5. Verify each target's position
target_time_valid = (
    ml_df["datetime_utc"]
    == ml_df["forecast_origin"]
       + pd.to_timedelta(ml_df["horizon"] - 1, unit="h")
).all()

print("\nTarget timestamps valid:", target_time_valid)

All windows have 24 hours: True
All horizons valid: True
Lag 24 leakage-free: True
Lag 48 leakage-free: True
Lag 168 leakage-free: True

Missing lag values:
lag_24     0
lag_48     0
lag_168    0
dtype: int64

Target timestamps valid: True


In [5]:
# Step 3.5: Rolling Feature Engineering

# Calculate historical rolling statistics
# shift(1) ensures the current hour is excluded

df["rolling_mean_24"] = (
    df["mw"].shift(1).rolling(window=24).mean()
)

df["rolling_std_24"] = (
    df["mw"].shift(1).rolling(window=24).std()
)

df["rolling_mean_168"] = (
    df["mw"].shift(1).rolling(window=168).mean()
)

df["rolling_std_168"] = (
    df["mw"].shift(1).rolling(window=168).std()
)

# Extract rolling features at each forecast origin
origin_features = df[
    [
        "datetime_utc",
        "rolling_mean_24",
        "rolling_std_24",
        "rolling_mean_168",
        "rolling_std_168"
    ]
].rename(columns={"datetime_utc": "forecast_origin"})

# Merge the same origin-level features into all 24 horizons
ml_df = ml_df.merge(
    origin_features,
    on="forecast_origin",
    how="left",
    validate="many_to_one"
)

print("Updated ML dataset shape:", ml_df.shape)

print("\nMissing rolling features:")
print(
    ml_df[
        [
            "rolling_mean_24",
            "rolling_std_24",
            "rolling_mean_168",
            "rolling_std_168"
        ]
    ].isna().sum()
)

display(
    ml_df[
        [
            "forecast_origin",
            "horizon",
            "lag_24",
            "lag_168",
            "rolling_mean_24",
            "rolling_std_24",
            "rolling_mean_168",
            "rolling_std_168",
            "mw"
        ]
    ].head()
)

Updated ML dataset shape: (43656, 21)

Missing rolling features:
rolling_mean_24     0
rolling_std_24      0
rolling_mean_168    0
rolling_std_168     0
dtype: int64


,forecast_origin,horizon,lag_24,lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168,rolling_std_168,mw
0,2021-01-08 05:00:00+00:00,1,87118.924,81842.644,96161.407042,6641.67699,90914.9365,8776.643656,88192.784
1,2021-01-08 05:00:00+00:00,2,85351.434,80005.690,96161.407042,6641.67699,90914.9365,8776.643656,86488.512
2,2021-01-08 05:00:00+00:00,3,84762.492,78940.704,96161.407042,6641.67699,90914.9365,8776.643656,85853.980
3,2021-01-08 05:00:00+00:00,4,84939.037,78563.968,96161.407042,6641.67699,90914.9365,8776.643656,85901.332
4,2021-01-08 05:00:00+00:00,5,87058.963,79136.453,96161.407042,6641.67699,90914.9365,8776.643656,87372.680


In [7]:
# Step 3.6: Define ML feature sets

calendar_features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "horizon"
]

lag_features = [
    "lag_24",
    "lag_48",
    "lag_168"
]

rolling_features = [
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168",
    "rolling_std_168"
]

feature_sets = {
    "A_Calendar": calendar_features,
    "B_Calendar_Lag": calendar_features + lag_features,
    "C_Full": calendar_features + lag_features + rolling_features
}

# Check feature availability
for name, features in feature_sets.items():
    missing_cols = set(features) - set(ml_df.columns)

    assert not missing_cols, f"{name}: Missing {missing_cols}"

    print(f"{name}: {len(features)} features")

# Check missing values in final features and target
final_features = feature_sets["C_Full"]

print("\nMissing values:")
print(ml_df[final_features + ["mw"]].isna().sum())

# Confirm sample counts
print("\nFinal ML Dataset:", ml_df.shape)
print("Forecast origins:", ml_df["forecast_origin"].nunique())

A_Calendar: 5 features
B_Calendar_Lag: 8 features
C_Full: 12 features

Missing values:
hour                0
day_of_week         0
month               0
is_weekend          0
horizon             0
lag_24              0
lag_48              0
lag_168             0
rolling_mean_24     0
rolling_std_24      0
rolling_mean_168    0
rolling_std_168     0
mw                  0
dtype: int64

Final ML Dataset: (43656, 21)
Forecast origins: 1819


In [8]:
# Step 3.7: Save final ML dataset

output_dir = project / "data_processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "pjm_rto_ml_features.csv"

ml_df = ml_df.sort_values(
    ["forecast_origin", "horizon"]
).reset_index(drop=True)

ml_df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Dataset shape:", ml_df.shape)

Saved: /Users/jamella/Desktop/27fall/ml/data_source/electricity-demand-forecasting/data_processed/pjm_rto_ml_features.csv
Dataset shape: (43656, 21)
